[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/02-data-science/03_pandas-basics.ipynb)

*Python for Data Science · Notebook 3 of 8*

# Pandas Basics

Series and DataFrames: building tables, selecting columns and rows, filtering, sorting and adding columns.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A DataFrame is a spreadsheet with superpowers: labelled columns, each a NumPy array underneath.

**What interviewers really test here:** loc vs iloc, selecting and filtering, and the SettingWithCopy trap.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Series and DataFrame *(Basics)*
- [ ] 2. Loading and looking at data *(Basics)*
- [ ] 3. A first look at any new dataset *(Basics)*
- [ ] 4. Selecting columns and rows *(Core)*
- [ ] 5. Filtering and sorting *(Core)*
- [ ] 6. loc vs iloc: labels vs positions *(Core)*
- [ ] 7. Filtering, step by step *(Core)*
- [ ] 8. New columns, apply and the index *(Advanced)*
- [ ] 9. Changing data safely *(Advanced)*
- [ ] 10. Vectorise before you apply *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### What pandas is

**pandas** is the standard Python library for working with **tabular** (spreadsheet- or SQL-like) and **time-series** data. It provides fast, labelled data structures built on NumPy arrays, plus tools to read, clean, transform, aggregate and reshape data.



The name comes from "panel data". Typical workflow: **load** (read\_csv) → **inspect** (head, info, describe) → **clean** → **transform** → **aggregate** (groupby) → **visualise / export**.

### Core data structures

|   | Series | DataFrame |
| --- | --- | --- |
| Dimensions | 1-D | 2-D |
| Made of | values + index | columns (Series) sharing one row index |
| dtype | one | one per column |
| Like | a labelled array / a column | a spreadsheet / SQL table |



An **Index** is an immutable sequence of labels (default `RangeIndex` 0, 1, 2 …). Labels need not be unique, but unique indexes make lookups fast. **Alignment**: arithmetic between Series/DataFrames matches rows by label; unmatched labels become NaN.

### Data types

| pandas dtype | Holds |
| --- | --- |
| `int64`, `float64` | numbers (NaN forces float) |
| `Int64` (capital I) | nullable integers (can hold `pd.NA`) |
| `bool` / `boolean` | True/False |
| `object` / `string` (`str` in pandas 3) | text |
| `category` | repeated labels stored as codes (saves memory) |
| `datetime64[ns]`, `timedelta64` | dates/times and durations |



Convert with `astype`, `pd.to_numeric`, `pd.to_datetime`.

### Selection and indexing

| Accessor | Selects by | Slice end |
| --- | --- | --- |
| `df[col]`, `df[[cols]]` | column name(s) | — |
| `df[mask]` | boolean Series (rows) | — |
| `df.loc[rows, cols]` | labels / boolean masks | **inclusive** |
| `df.iloc[rows, cols]` | integer positions | exclusive |
| `df.at[r, c]`, `df.iat[i, j]` | a single cell (fast) | — |



**Chained assignment** like `df[df.a > 0]["b"] = 1` may modify a temporary copy (SettingWithCopyWarning). Use one `.loc` call: `df.loc[df.a > 0, "b"] = 1`. pandas 3 uses Copy-on-Write, so chained assignment never changes the original.

### I/O and inspection

| Read | Write |
| --- | --- |
| `pd.read_csv(path, sep=",", usecols=…, parse_dates=…)` | `df.to_csv(path, index=False)` |
| `pd.read_excel` | `df.to_excel` |
| `pd.read_json` | `df.to_json` |
| `pd.read_sql(query, conn)` | `df.to_sql` |
| `pd.read_parquet` | `df.to_parquet` (fast, typed, compressed) |



Inspect with `head, tail, sample, shape, info, dtypes, describe, columns, index, value_counts, nunique, isna().sum()`.

### Vectorised operations vs apply

Column arithmetic, comparisons and the `.str` / `.dt` accessors run vectorised on whole columns. `apply`, `map` with a Python function and `iterrows` loop in Python and are much slower on large data. Order of preference: built-in vectorised method → `np.where` / `pd.cut` / `map` with a dict → `apply` → explicit loops.

---
# Part 2: Lessons, step by step

## Lesson 1: Series and DataFrame

*Level: Basics*

Pandas adds two labelled data structures on top of NumPy:




- A **Series** is a single column: values plus an **index** (the row labels).
- A **DataFrame** is a table: several Series sharing the same index, one per column. Each column has its own dtype.



```python
import pandas as pd
s = pd.Series([82, 45, 67], index=["Ana", "Ben", "Cy"])
df = pd.DataFrame({"name": ["Ana", "Ben", "Cy"],
                   "score": [82, 45, 67]})
```



In the animations, DataFrames are drawn as tables with the index on the left and column names on top; a Series is drawn as a row of boxes with its index under each value.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def structures_demo():
    s = pd.Series([82, 45, 67], index=["Ana", "Ben", "Cy"])
    df = pd.DataFrame({"name": ["Ana", "Ben", "Cy"], "score": [82, 45, 67]})
    print(s["Ben"], df.shape, list(df.columns))
    print(df)


structures_demo()

45 (3, 2) ['name', 'score']
  name  score
0  Ana     82
1  Ben     45
2   Cy     67


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 2: Loading and looking at data

*Level: Basics*

Real data usually comes from a file: `pd.read_csv("sales.csv")`, `pd.read_excel`, `pd.read_json`, `pd.read_sql`. Then look before you analyse:



| Call | Shows |
| --- | --- |
| `df.head(5)` / `df.tail()` | first / last rows |
| `df.shape` | (rows, columns) |
| `df.columns`, `df.dtypes` | column names and types |
| `df.info()` | types, non-null counts, memory |
| `df.describe()` | count, mean, std, min, quartiles, max of numeric columns |
| `df["col"].value_counts()` | frequency of each category |

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
import io


def loading_demo():
    text = "name,city,score\nAna,Pune,82\nBen,Delhi,45\nCy,Pune,67\n"
    df = pd.read_csv(io.StringIO(text))
    summary = df["score"].describe().round(1)
    counts = df["city"].value_counts()
    print(df.head(2))
    print(summary["mean"], summary["max"], counts["Pune"])


loading_demo()

  name   city  score
0  Ana   Pune     82
1  Ben  Delhi     45
64.7 82.0 2


**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q11](#q11), [Q9](#q9), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 3: A first look at any new dataset

*Level: Basics*

Run the same few checks every time, before any analysis:



| Code | Answers |
| --- | --- |
| `df.shape` | how many rows and columns? |
| `df.head()` | what does a row look like? |
| `df.dtypes` | are numbers stored as numbers? Is any date stored as text? |
| `df.isna().sum()` | where are the gaps? |
| `df.describe()` | any impossible values, such as age 300? |
| `df["city"].value_counts()` | are categories spelled consistently? |
| `df.duplicated().sum()` | any repeated rows? |



#### What the demo's data reveals



- `price` has one gap.
- `describe` shows a max age of 200, which must be an error.
- "pune" and "Pune" count as two cities.



Five minutes here prevents wrong conclusions later.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def first_look():
    df = pd.DataFrame({"city": ["Pune", "pune", "Delhi", "Pune"], "age": [25, 31, 200, 40], "price": [100.0, None, 300.0, 150.0]})
    shape = df.shape
    gaps = df.isna().sum()
    summary = df["age"].describe()
    cities = df["city"].value_counts()
    fixed_cities = df["city"].str.title().value_counts()


first_look()

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q11](#q11), [Q9](#q9), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 4: Selecting columns and rows

*Level: Core*

| Code | Selects |
| --- | --- |
| `df["score"]` | one column (Series) |
| `df[["name", "score"]]` | several columns (DataFrame) |
| `df.loc[label]`, `df.loc[label, "col"]` | by index **label** |
| `df.iloc[0]`, `df.iloc[0, 1]`, `df.iloc[:3]` | by **position** |
| `df.loc[2:4]` | labels 2 to 4 **inclusive** |
| `df.iloc[2:4]` | positions 2 and 3 (end excluded, like Python) |



`set_index("name")` makes a column the index, so `df.loc["Ana"]` finds Ana's row.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def selecting_demo():
    df = pd.DataFrame({"name": ["Ana", "Ben", "Cy"], "score": [82, 45, 67], "city": ["Pune", "Delhi", "Pune"]})
    scores = df["score"]
    two_cols = df[["name", "city"]]
    by_name = df.set_index("name")
    ben = by_name.loc["Ben", "score"]
    first_row = df.iloc[0]
    print(ben, first_row["name"], two_cols.shape)


selecting_demo()

45 Ana (3, 2)


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q4](#q4), [Q17](#q17), [Q8](#q8), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 5: Filtering and sorting

*Level: Core*

A comparison on a column gives a boolean Series (a **mask**); indexing with it keeps the matching rows:



```python
df[df["score"] > 60]
df[(df["score"] > 60) & (df["city"] == "Pune")]   # and → &, or → |, not → ~
df[df["city"].isin(["Pune", "Goa"])]
df.query("score > 60 and city == 'Pune'")         # same, as a string
df.sort_values("score", ascending=False)
df.sort_values(["city", "score"], ascending=[True, False])
df.nlargest(3, "score")
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def filtering_demo():
    df = pd.DataFrame({"name": ["Ana", "Ben", "Cy", "Dia"], "score": [82, 45, 67, 91], "city": ["Pune", "Delhi", "Pune", "Goa"]})
    mask = (df["score"] > 60) & (df["city"] == "Pune")
    pune_good = df[mask]
    ranked = df.sort_values("score", ascending=False)
    print(pune_good["name"].tolist(), ranked["name"].tolist())


filtering_demo()

['Ana', 'Cy'] ['Dia', 'Ana', 'Cy', 'Ben']


**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q18](#q18), [Q14](#q14), [Q6](#q6), [Q20](#q20), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 6: loc vs iloc: labels vs positions

*Level: Core*

A DataFrame has labels (the index) and positions (0, 1, 2…). They look alike until they differ.



```python
df = pd.DataFrame({"score": [50, 60, 70]}, index=[10, 20, 30])
```



| Code | Means | Rows returned |
| --- | --- | --- |
| `df.loc[10:20]` | labels 10 to 20, **end included** | 50, 60 |
| `df.iloc[0:2]` | positions 0 and 1, **end excluded** | 50, 60 |
| `df.loc[20, "score"]` | label 20, column score | 60 |
| `df.iloc[1, 0]` | row position 1, column position 0 | 60 |



#### The trap after filtering



Filtering keeps the original labels. After `high = df[df.score > 50]`, the labels are 20 and 30. So `high.loc[10]` fails, while `high.iloc[0]` gives the first remaining row.



Use `.reset_index(drop=True)` to renumber from 0.



**Rule of thumb:** use `loc` with column names and conditions. Use `iloc` only for "the first 5 rows".

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def loc_iloc_demo():
    df = pd.DataFrame({"score": [50, 60, 70]}, index=[10, 20, 30])
    by_label = df.loc[10:20]
    by_position = df.iloc[0:2]
    high = df[df["score"] > 50]
    first_remaining = high.iloc[0]
    renumbered = high.reset_index(drop=True)


loc_iloc_demo()

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q3](#q3), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 7: Filtering, step by step

*Level: Core*

A filter is two steps: build a True/False column, then use it to pick rows.



```python
mask = df["score"] > 60        # one True or False per row
df[mask]                       # only the True rows
```



| Goal | Code |
| --- | --- |
| Both conditions | `df[(df.score > 60) & (df.city == "Pune")]` |
| Either condition | `df[(df.score > 60) \| (df.city == "Pune")]` |
| Not | `df[~(df.city == "Pune")]` |
| One of several values | `df[df.city.isin(["Pune", "Goa"])]` |
| A range, ends included | `df[df.score.between(60, 80)]` |
| Text contains | `df[df.name.str.contains("an")]` |
| Readable form | `df.query("score > 60 and city == 'Pune'")` |



#### The two most common errors



- Using `and`/`or` gives "truth value of a Series is ambiguous". Use `&` and `|`.
- Leaving out brackets: `df.a > 1 & df.b > 2` is read as `df.a > (1 & df.b) > 2`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [8]:
def filter_steps():
    df = pd.DataFrame({"name": ["Ana", "Ben", "Cy", "Dee"], "city": ["Pune", "Delhi", "Goa", "Pune"], "score": [72, 45, 90, 58]})
    mask = df["score"] > 60
    high = df[mask]
    both = df[(df["score"] > 50) & (df["city"] == "Pune")]
    listed = df[df["city"].isin(["Pune", "Goa"])]
    in_range = df[df["score"].between(50, 75)]
    readable = df.query("score > 50 and city == 'Pune'")


filter_steps()

**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q18](#q18), [Q14](#q14), [Q20](#q20), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 8: New columns, apply and the index

*Level: Advanced*

Create or overwrite a column by assigning to it — the maths runs on whole columns at once:



```python
df["total"] = df["price"] * df["qty"]
df["passed"] = df["score"] >= 40
df["grade"] = df["score"].apply(lambda s: "A" if s >= 80 else "B")
df["city"] = df["city"].map({"Pune": "PN", "Goa": "GA"})
df = df.assign(bonus=df["score"] * 0.1)
```



Prefer vectorised column maths; `apply` with a Python function runs a loop under the hood and is slower.



The **index** labels rows. Operations between Series **align on the index**, not on position — a common source of surprising NaNs. `reset_index()` turns the index back into a column.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [9]:
def new_columns_demo():
    df = pd.DataFrame({"item": ["pen", "ink", "cap"], "price": [10, 25, 5], "qty": [3, 2, 4]})
    df["total"] = df["price"] * df["qty"]
    df["big"] = df["total"] >= 40
    df["label"] = df["item"].apply(str.upper)
    a = pd.Series([1, 2], index=["x", "y"])
    b = pd.Series([10, 20], index=["y", "z"])
    print(df)
    print(a + b)


new_columns_demo()

  item  price  qty  total    big label
0  pen     10    3     30  False   PEN
1  ink     25    2     50   True   INK
2  cap      5    4     20  False   CAP
x     NaN
y    12.0
z     NaN
dtype: float64


**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q15](#q15), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 9: Changing data safely

*Level: Advanced*

This goes wrong for many beginners:



```python
high = df[df.score > 60]
high["grade"] = "A"        # does this change df? No!
```



A filtered result is its own table, and changing it doesn't reach back to `df`. Older pandas warns (`SettingWithCopyWarning`), and in newer pandas it never changes the original.



#### Choose what you mean



| You want | Write |
| --- | --- |
| To change the original | `df.loc[df.score > 60, "grade"] = "A"` |
| A separate table to edit | `high = df[df.score > 60].copy()` |



#### Never chain assignments



`df[df.score > 60]["grade"] = "A"` does nothing useful. Keep the row condition and the column inside one `.loc[...]`.



Most methods return a new table: `df.sort_values("score")` leaves `df` unchanged until you assign the result back.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [10]:
def safe_changes():
    df = pd.DataFrame({"score": [72, 45, 90]})
    high = df[df["score"] > 60].copy()
    high["grade"] = "A"
    original_has_grade = "grade" in df.columns
    df.loc[df["score"] > 60, "grade"] = "A"
    sorted_copy = df.sort_values("score")
    first_row_still = df.iloc[0, 0]


safe_changes()

**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q13](#q13), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 10: Vectorise before you apply

*Level: Advanced*

`df.apply` runs Python once per row, so it's slow on large tables. Prefer a built-in where one exists.



| Task | Slow | Better |
| --- | --- | --- |
| Add columns | `apply(lambda r: r.a + r.b, axis=1)` | `df.a + df.b` |
| Two outcomes | a function with if/else | `np.where(df.a > 50, "pass", "fail")` |
| Many outcomes | a long if/elif | `np.select(conditions, choices)` or `pd.cut` |
| Lookup | a function with a dict | `df.city.map(lookup)` |
| Text | `apply(lambda s: s.lower())` | `df.name.str.lower()` |



#### Worked example: grades



With marks 35, 55, 72, 90, `pd.cut(marks, [0, 50, 70, 100], labels=["F", "C", "A"])` returns F, C, A, A. Each bin includes its right edge, so a mark of exactly 50 would be F and exactly 70 would be C.



Use `apply` when the logic really can't be expressed any other way. It's fine on small data.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [11]:
def faster_columns():
    df = pd.DataFrame({"marks": [35, 55, 72, 90], "city": ["P", "D", "P", "G"]})
    two_way = np.where(df["marks"] >= 50, "pass", "fail")
    binned = pd.cut(df["marks"], [0, 50, 70, 100], labels=["F", "C", "A"]).astype(str)
    conditions = [df["marks"] >= 70, df["marks"] >= 50]
    many_way = np.select(conditions, ["A", "C"], default="F")
    looked_up = df["city"].map({"P": "Pune", "D": "Delhi", "G": "Goa"})


faster_columns()

**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q19](#q19), [Q16](#q16). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Rows and Columns

🟢 Warm-up · *DataFrame · shape*

Build a table from a dict of equal-length lists and return `(rows, columns)`.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}` → `(4, 3)`
- `data = {'a': [1]}` → `(1, 1)`

In [ ]:
def table_shape(data):
    """
    Build a table from a dict of equal-length lists and return (rows, columns).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(table_shape, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']},), (4, 3)),
    (({'a': [1]},), (1, 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count the keys for columns and the length of any list for rows.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Create a DataFrame and read `df.shape`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Count the keys for columns and the length of any list for rows.

```python
def table_shape(data):
    cols = len(data)
    rows = len(next(iter(data.values())))
    return (rows, cols)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Create a DataFrame and read `df.shape`.

```python
def table_shape(data):
    df = pd.DataFrame(data)
    return df.shape
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count the keys for columns and the length of any list for rows. That costs O(1) time and O(1) space."

"I can do better: Create a DataFrame and read `df.shape`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/rows-and-columns)

<a id="q2"></a>
### Q2. Column Names

🟢 Warm-up · *df.columns*

Return the column names of a table given as a list of row dicts, in first-seen order.

**Examples**

- `rows = [{'a': 1, 'b': 2}, {'a': 3, 'c': 4}]` → `['a', 'b', 'c']`
- `rows = [{'x': 1}]` → `['x']`

In [ ]:
def column_names(rows):
    """
    Return the column names of a table given as a list of row dicts, in first-
    seen order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(column_names, [
    (([{'a': 1, 'b': 2}, {'a': 3, 'c': 4}],), ['a', 'b', 'c']),
    (([{'x': 1}],), ['x']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk every row and remember keys you haven't seen.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pd.DataFrame(rows).columns` — pandas aligns the keys for you (missing cells become NaN).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Walk every row and remember keys you haven't seen.

```python
def column_names(rows):
    names = []
    for row in rows:
        for key in row:
            if key not in names:
                names.append(key)
    return names
```

**Cost:** O(cells) time, O(cols) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pd.DataFrame(rows).columns` — pandas aligns the keys for you (missing cells become NaN).

```python
def column_names(rows):
    df = pd.DataFrame(rows)
    return df.columns.tolist()
```

**Cost:** O(cells) time, O(cells) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk every row and remember keys you haven't seen. That costs O(cells) time and O(cols) space."

"I can do better: `pd.DataFrame(rows).columns` — pandas aligns the keys for you (missing cells become NaN). That's O(cells) time and O(cells) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/column-names)

<a id="q3"></a>
### Q3. Pick a Column

🟢 Warm-up · *df[col] → Series*

Return the values of one column as a list.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, col = 'score'` → `[82, 45, 67, 91]`
- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, col = 'city'` → `['Pune', 'Delhi', 'Pune', 'Goa']`

In [ ]:
def column_values(data, col):
    """
    Return the values of one column as a list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(column_values, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 'score'), [82, 45, 67, 91]),
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 'city'), ['Pune', 'Delhi', 'Pune', 'Goa']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Just take the list from the dict (or loop over row dicts).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df[col]` gives a Series; `.tolist()` turns it into a list.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Just take the list from the dict (or loop over row dicts).

```python
def column_values(data, col):
    rows = [dict(zip(data, values)) for values in zip(*data.values())]
    return [row[col] for row in rows]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df[col]` gives a Series; `.tolist()` turns it into a list.

```python
def column_values(data, col):
    df = pd.DataFrame(data)
    return df[col].tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Just take the list from the dict (or loop over row dicts). That costs O(n) time and O(n) space."

"I can do better: `df[col]` gives a Series; `.tolist()` turns it into a list. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/pick-a-column)

<a id="q4"></a>
### Q4. Pick Several Columns

🟢 Warm-up · *df[[...]]*

Return a table with only the requested columns, in the requested order.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, cols = ['city', 'name']` → `{'city': ['Pune', 'Delhi', 'Pune', 'Goa'], 'name': ['Ana', 'Ben', 'Cy', 'Dia']}`

In [ ]:
def some_columns(data, cols):
    """
    Return a table with only the requested columns, in the requested order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(some_columns, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, ['city', 'name']), {'city': ['Pune', 'Delhi', 'Pune', 'Goa'], 'name': ['Ana', 'Ben', 'Cy', 'Dia']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a new dict with just those keys.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Double brackets select a list of columns: `df[['city', 'name']]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Build a new dict with just those keys.

```python
def some_columns(data, cols):
    out = {}
    for c in cols:
        out[c] = list(data[c])
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Double brackets select a list of columns: `df[['city', 'name']]`.

```python
def some_columns(data, cols):
    df = pd.DataFrame(data)
    return df[cols]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a new dict with just those keys. That costs O(n) time and O(n) space."

"I can do better: Double brackets select a list of columns: `df[['city', 'name']]`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/pick-several-columns)

<a id="q5"></a>
### Q5. Filter Rows

🟢 Warm-up · *Boolean filtering*

Return the names of everyone with a score above `cut`.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, cut = 60` → `['Ana', 'Cy', 'Dia']`
- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, cut = 95` → `[]`

In [ ]:
def above(data, cut):
    """
    Return the names of everyone with a score above cut.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(above, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 60), ['Ana', 'Cy', 'Dia']),
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 95), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over positions and test each score.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A boolean Series selects rows: `df.loc[df['score'] > cut, 'name']`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop over positions and test each score.

```python
def above(data, cut):
    out = []
    for i in range(len(data["name"])):
        if data["score"][i] > cut:
            out.append(data["name"][i])
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** A boolean Series selects rows: `df.loc[df['score'] > cut, 'name']`.

```python
def above(data, cut):
    df = pd.DataFrame(data)
    return df.loc[df["score"] > cut, "name"].tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over positions and test each score. That costs O(n) time and O(n) space."

"I can do better: A boolean Series selects rows: `df.loc[df['score'] > cut, 'name']`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/filter-rows)

<a id="q6"></a>
### Q6. Top n by Score

🟢 Warm-up · *sort\_values / nlargest*

Return the names of the n highest scorers, best first.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, n = 2` → `['Dia', 'Ana']`
- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, n = 1` → `['Dia']`

In [ ]:
def top_n(data, n):
    """
    Return the names of the n highest scorers, best first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_n, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 2), ['Dia', 'Ana']),
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 1), ['Dia']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Zip names with scores, sort by score descending, take n.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.nlargest(n, 'score')['name']`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Zip names with scores, sort by score descending, take n.

```python
def top_n(data, n):
    pairs = list(zip(data["score"], data["name"]))
    pairs.sort(reverse=True)
    return [name for score, name in pairs[:n]]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.nlargest(n, 'score')['name']`.

```python
def top_n(data, n):
    df = pd.DataFrame(data)
    return df.nlargest(n, "score")["name"].tolist()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Zip names with scores, sort by score descending, take n. That costs O(n log n) time and O(n) space."

"I can do better: `df.nlargest(n, 'score')['name']`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/top-n-by-score)

<a id="q7"></a>
### Q7. Add a Computed Column

🟢 Warm-up · *Vectorised column maths*

Add a column `total = price × qty` and return the table.

**Examples**

- `data = {'item': ['pen', 'ink'], 'price': [10, 25], 'qty': [3, 2]}` → `{'item': ['pen', 'ink'], 'price': [10, 25], 'qty': [3, 2], 'total': [30, 50]}`

In [ ]:
def with_total(data):
    """
    Add a column total = price × qty and return the table.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(with_total, [
    (({'item': ['pen', 'ink'], 'price': [10, 25], 'qty': [3, 2]},), {'item': ['pen', 'ink'], 'price': [10, 25], 'qty': [3, 2], 'total': [30, 50]}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over positions computing each total.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df['total'] = df['price'] * df['qty']` — whole columns at once.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop over positions computing each total.

```python
def with_total(data):
    out = {k: list(v) for k, v in data.items()}
    out["total"] = []
    for i in range(len(out["price"])):
        out["total"].append(out["price"][i] * out["qty"][i])
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df['total'] = df['price'] * df['qty']` — whole columns at once.

```python
def with_total(data):
    df = pd.DataFrame(data)
    df["total"] = df["price"] * df["qty"]
    return df
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over positions computing each total. That costs O(n) time and O(n) space."

"I can do better: `df['total'] = df['price'] * df['qty']` — whole columns at once. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/add-a-computed-column)

<a id="q8"></a>
### Q8. Rename Columns

🟢 Warm-up · *rename*

Rename columns using the mapping (old → new) and return the new column list.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, mapping = {'name': 'student', 'city': 'town'}` → `['student', 'score', 'town']`

In [ ]:
def renamed(data, mapping):
    """
    Rename columns using the mapping (old → new) and return the new column list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(renamed, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, {'name': 'student', 'city': 'town'}), ['student', 'score', 'town']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Map each key through the dict by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.rename(columns=mapping)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Map each key through the dict by hand.

```python
def renamed(data, mapping):
    return [mapping.get(c, c) for c in data]
```

**Cost:** O(cols) time, O(cols) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.rename(columns=mapping)`.

```python
def renamed(data, mapping):
    df = pd.DataFrame(data).rename(columns=mapping)
    return df.columns.tolist()
```

**Cost:** O(cols) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Map each key through the dict by hand. That costs O(cols) time and O(cols) space."

"I can do better: `df.rename(columns=mapping)`. That's O(cols) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/rename-columns)

<a id="q9"></a>
### Q9. Count Each Category

🟢 Warm-up · *value\_counts*

Return how many rows have each city, as {city: count}, most common first.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}` → `{'Pune': 2, 'Delhi': 1, 'Goa': 1}`

In [ ]:
def city_counts(data):
    """
    Return how many rows have each city, as {city: count}, most common first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(city_counts, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']},), {'Pune': 2, 'Delhi': 1, 'Goa': 1}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count with a dict, then sort by count.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df['city'].value_counts()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Count with a dict, then sort by count.

```python
def city_counts(data):
    counts = {}
    for c in data["city"]:
        counts[c] = counts.get(c, 0) + 1
    return dict(sorted(counts.items(), key=lambda kv: -kv[1]))
```

**Cost:** O(n log n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df['city'].value_counts()`.

```python
def city_counts(data):
    df = pd.DataFrame(data)
    return df["city"].value_counts()
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count with a dict, then sort by count. That costs O(n log n) time and O(k) space."

"I can do better: `df['city'].value_counts()`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/count-each-category)

<a id="q10"></a>
### Q10. How Many Different Values

🟢 Warm-up · *nunique*

Return the number of distinct values in column `col`.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, col = 'city'` → `3`
- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, col = 'name'` → `4`

In [ ]:
def distinct(data, col):
    """
    Return the number of distinct values in column col.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(distinct, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 'city'), 3),
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 'name'), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Put the values in a set and take its size.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df[col].nunique()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Put the values in a set and take its size.

```python
def distinct(data, col):
    return len(set(data[col]))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df[col].nunique()`.

```python
def distinct(data, col):
    return pd.DataFrame(data)[col].nunique()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Put the values in a set and take its size. That costs O(n) time and O(n) space."

"I can do better: `df[col].nunique()`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/how-many-different-values)

<a id="q11"></a>
### Q11. Quick Summary of a Column

🟢 Warm-up · *describe / agg*

Return {'mean', 'min', 'max'} of the score column (mean rounded to 2).

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}` → `{'mean': 71.25, 'min': 45, 'max': 91}`

In [ ]:
def score_summary(data):
    """
    Return {'mean', 'min', 'max'} of the score column (mean rounded to 2).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(score_summary, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']},), {'mean': 71.25, 'min': 45, 'max': 91}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute each statistic by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df['score'].agg(['mean', 'min', 'max'])`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Compute each statistic by hand.

```python
def score_summary(data):
    s = data["score"]
    return {"mean": round(sum(s) / len(s), 2), "min": min(s), "max": max(s)}
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df['score'].agg(['mean', 'min', 'max'])`.

```python
def score_summary(data):
    stats = pd.DataFrame(data)["score"].agg(["mean", "min", "max"])
    return {"mean": round(stats["mean"], 2), "min": stats["min"], "max": stats["max"]}
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute each statistic by hand. That costs O(n) time and O(1) space."

"I can do better: `df['score'].agg(['mean', 'min', 'max'])`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/quick-summary-of-a-column)

<a id="q12"></a>
### Q12. Read CSV Text

🟢 Warm-up · *read\_csv*

Given the text of a CSV file, return `(rows, columns, column names)`.

**Examples**

- `text = 'name,age\nAna,21\nBen,25\n'` → `(2, 2, ['name', 'age'])`
- `text = 'x\n1\n2\n3\n'` → `(3, 1, ['x'])`

In [ ]:
import io

def csv_info(text):
    """
    Given the text of a CSV file, return (rows, columns, column names).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(csv_info, [
    (('name,age\nAna,21\nBen,25\n',), (2, 2, ['name', 'age'])),
    (('x\n1\n2\n3\n',), (3, 1, ['x'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split the text into lines and the first line on commas.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pd.read_csv` parses it (here from `io.StringIO`, normally from a file path).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Split the text into lines and the first line on commas.

```python
def csv_info(text):
    lines = [l for l in text.strip().split("\n") if l]
    header = lines[0].split(",")
    return (len(lines) - 1, len(header), header)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pd.read_csv` parses it (here from `io.StringIO`, normally from a file path).

```python
def csv_info(text):
    import io
    df = pd.read_csv(io.StringIO(text))
    return (len(df), len(df.columns), df.columns.tolist())
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split the text into lines and the first line on commas. That costs O(n) time and O(n) space."

"I can do better: `pd.read_csv` parses it (here from `io.StringIO`, normally from a file path). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/read-csv-text)

<a id="q13"></a>
### Q13. Drop a Column

🟢 Warm-up · *drop*

Return the column names after removing `col`.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, col = 'city'` → `['name', 'score']`

In [ ]:
def without_column(data, col):
    """
    Return the column names after removing col.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(without_column, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 'city'), ['name', 'score']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

List comprehension over the keys.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.drop(columns=col)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** List comprehension over the keys.

```python
def without_column(data, col):
    return [c for c in data if c != col]
```

**Cost:** O(cols) time, O(cols) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.drop(columns=col)`.

```python
def without_column(data, col):
    return pd.DataFrame(data).drop(columns=col).columns.tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: List comprehension over the keys. That costs O(cols) time and O(cols) space."

"I can do better: `df.drop(columns=col)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/drop-a-column)

<a id="q14"></a>
### Q14. Rows From a List of Values

🟢 Warm-up · *isin*

Return the names of people living in any of the given cities.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, cities = ['Goa', 'Delhi']` → `['Ben', 'Dia']`
- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, cities = []` → `[]`

In [ ]:
def in_cities(data, cities):
    """
    Return the names of people living in any of the given cities.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(in_cities, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, ['Goa', 'Delhi']), ['Ben', 'Dia']),
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, []), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and check `city in cities`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df[df['city'].isin(cities)]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop and check `city in cities`.

```python
def in_cities(data, cities):
    return [n for n, c in zip(data["name"], data["city"]) if c in cities]
```

**Cost:** O(n·k) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df[df['city'].isin(cities)]`.

```python
def in_cities(data, cities):
    df = pd.DataFrame(data)
    return df.loc[df["city"].isin(cities), "name"].tolist()
```

**Cost:** O(n + k) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and check `city in cities`. That costs O(n·k) time and O(n) space."

"I can do better: `df[df['city'].isin(cities)]`. That's O(n + k) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/rows-from-a-list-of-values)

<a id="q15"></a>
### Q15. Letter Grades Column

🟢 Warm-up · *apply / map*

Return the letter grade for every score: ≥ 80 'A', ≥ 60 'B', else 'C'.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}` → `['A', 'C', 'B', 'A']`

In [ ]:
def grades(data):
    """
    Return the letter grade for every score: ≥ 80 'A', ≥ 60 'B', else 'C'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(grades, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']},), ['A', 'C', 'B', 'A']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with if/elif.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Write a small function and `.apply` it to the column (or use `pd.cut`).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop with if/elif.

```python
def grades(data):
    out = []
    for s in data["score"]:
        if s >= 80:
            out.append("A")
        elif s >= 60:
            out.append("B")
        else:
            out.append("C")
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Write a small function and `.apply` it to the column (or use `pd.cut`).

```python
def grades(data):
    def letter(s):
        return "A" if s >= 80 else "B" if s >= 60 else "C"
    return pd.DataFrame(data)["score"].apply(letter).tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with if/elif. That costs O(n) time and O(n) space."

"I can do better: Write a small function and `.apply` it to the column (or use `pd.cut`). That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}`

1. letter() returns 'A'
2. letter() returns 'C'
3. letter() returns 'B'
4. letter() returns 'A'
5. grades() returns ['A', 'C', 'B', 'A']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/letter-grades-column)

<a id="q16"></a>
### Q16. Mean Score per Row Condition

🟢 Warm-up · *Boolean mask + aggregation*

Return the average score of people from `city`, rounded to 2 decimals (0 if nobody).

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, city = 'Pune'` → `74.5`
- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, city = 'Oslo'` → `0`

In [ ]:
def city_average(data, city):
    """
    Return the average score of people from city, rounded to 2 decimals (0 if
    nobody).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(city_average, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 'Pune'), 74.5),
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 'Oslo'), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect matching scores in a loop, then average.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Filter with a mask, take the column and call `.mean()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Collect matching scores in a loop, then average.

```python
def city_average(data, city):
    scores = [s for s, c in zip(data["score"], data["city"]) if c == city]
    return round(sum(scores) / len(scores), 2) if scores else 0
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Filter with a mask, take the column and call `.mean()`.

```python
def city_average(data, city):
    df = pd.DataFrame(data)
    scores = df.loc[df["city"] == city, "score"]
    return round(scores.mean(), 2) if len(scores) else 0
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect matching scores in a loop, then average. That costs O(n) time and O(n) space."

"I can do better: Filter with a mask, take the column and call `.mean()`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/mean-score-per-row-condition)

<a id="q17"></a>
### Q17. loc vs iloc

🟡 Interview standard · *Label vs position*

With names as the index, return `(score by label name, score at position pos)`.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, name = 'Cy', pos = 0` → `(67, 82)`
- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, name = 'Ana', pos = 3` → `(82, 91)`

In [ ]:
def lookups(data, name, pos):
    """
    With names as the index, return (score by label name, score at position
    pos).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lookups, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 'Cy', 0), (67, 82)),
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 'Ana', 3), (82, 91)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Find the name's position with `index()`, and index the score list by position.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.loc[name, 'score']` uses the index label; `df.iloc[pos, col]` uses the position.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Find the name's position with `index()`, and index the score list by position.

```python
def lookups(data, name, pos):
    by_label = data["score"][data["name"].index(name)]
    by_position = data["score"][pos]
    return (by_label, by_position)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.loc[name, 'score']` uses the index label; `df.iloc[pos, col]` uses the position.

```python
def lookups(data, name, pos):
    df = pd.DataFrame(data).set_index("name")
    by_label = df.loc[name, "score"]
    by_position = df.iloc[pos, df.columns.get_loc("score")]
    return (by_label, by_position)
```

**Cost:** O(1) avg time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Find the name's position with `index()`, and index the score list by position. That costs O(n) time and O(1) space."

"I can do better: `df.loc[name, 'score']` uses the index label; `df.iloc[pos, col]` uses the position. That's O(1) avg time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/loc-vs-iloc)

<a id="q18"></a>
### Q18. Two Conditions

🟡 Interview standard · *& and |*

Count rows where score ≥ `cut` AND city equals `city`.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, cut = 60, city = 'Pune'` → `2`
- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, cut = 70, city = 'Pune'` → `1`

In [ ]:
def count_both(data, cut, city):
    """
    Count rows where score ≥ cut AND city equals city.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_both, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 60, 'Pune'), 2),
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}, 70, 'Pune'), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and test both conditions with `and`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Combine masks with `&` (each condition in brackets) and sum the True values.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop and test both conditions with `and`.

```python
def count_both(data, cut, city):
    count = 0
    for s, c in zip(data["score"], data["city"]):
        if s >= cut and c == city:
            count += 1
    return count
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Combine masks with `&` (each condition in brackets) and sum the True values.

```python
def count_both(data, cut, city):
    df = pd.DataFrame(data)
    mask = (df["score"] >= cut) & (df["city"] == city)
    return mask.sum()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and test both conditions with `and`. That costs O(n) time and O(1) space."

"I can do better: Combine masks with `&` (each condition in brackets) and sum the True values. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/two-conditions)

<a id="q19"></a>
### Q19. Best Subject per Student

🟡 Interview standard · *idxmax(axis=1)*

Rows are students, columns are subjects. Return {student: subject with the highest mark}.

**Examples**

- `data = {'name': ['Ana', 'Ben'], 'maths': [90, 60], 'art': [70, 85]}` → `{'Ana': 'maths', 'Ben': 'art'}`

In [ ]:
def best_subject(data):
    """
    Rows are students, columns are subjects. Return {student: subject with the
    highest mark}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(best_subject, [
    (({'name': ['Ana', 'Ben'], 'maths': [90, 60], 'art': [70, 85]},), {'Ana': 'maths', 'Ben': 'art'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each row, loop over subjects and track the best.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Make names the index and call `idxmax(axis=1)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** For each row, loop over subjects and track the best.

```python
def best_subject(data):
    subjects = [c for c in data if c != "name"]
    out = {}
    for i, name in enumerate(data["name"]):
        best = subjects[0]
        for s in subjects:
            if data[s][i] > data[best][i]:
                best = s
        out[name] = best
    return out
```

**Cost:** O(rows·cols) time, O(rows) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Make names the index and call `idxmax(axis=1)`.

```python
def best_subject(data):
    df = pd.DataFrame(data).set_index("name")
    return df.idxmax(axis=1)
```

**Cost:** O(rows·cols) time, O(rows) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each row, loop over subjects and track the best. That costs O(rows·cols) time and O(rows) space."

"I can do better: Make names the index and call `idxmax(axis=1)`. That's O(rows·cols) time and O(rows) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/best-subject-per-student)

<a id="q20"></a>
### Q20. Sort by Two Columns

🟡 Interview standard · *sort\_values with a list*

Sort by city A→Z, then by score high→low; return the names in that order.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']}` → `['Ben', 'Dia', 'Ana', 'Cy']`

In [ ]:
def sorted_names(data):
    """
    Sort by city A→Z, then by score high→low; return the names in that order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sorted_names, [
    (({'name': ['Ana', 'Ben', 'Cy', 'Dia'], 'score': [82, 45, 67, 91], 'city': ['Pune', 'Delhi', 'Pune', 'Goa']},), ['Ben', 'Dia', 'Ana', 'Cy']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

`sorted` with a tuple key (city, −score).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.sort_values(['city', 'score'], ascending=[True, False])`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** `sorted` with a tuple key (city, −score).

```python
def sorted_names(data):
    rows = list(zip(data["city"], data["score"], data["name"]))
    rows.sort(key=lambda r: (r[0], -r[1]))
    return [r[2] for r in rows]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.sort_values(['city', 'score'], ascending=[True, False])`.

```python
def sorted_names(data):
    df = pd.DataFrame(data).sort_values(["city", "score"], ascending=[True, False])
    return df["name"].tolist()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: `sorted` with a tuple key (city, −score). That costs O(n log n) time and O(n) space."

"I can do better: `df.sort_values(['city', 'score'], ascending=[True, False])`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-basics/sort-by-two-columns)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Series vs DataFrame?</b></summary>

A Series is a one-dimensional labelled array (one column). A DataFrame is a two-dimensional table of columns that share a row index; each column is a Series.

</details>

<details>
<summary><b>2. loc vs iloc?</b></summary>

`loc` selects by labels (and its slices include the end label). `iloc` selects by integer positions (slices exclude the end, like Python).

</details>

<details>
<summary><b>3. Why use & instead of and in filters?</b></summary>

`and` needs a single True/False, but a mask is a whole Series of booleans. `&`, `|`, `~` combine masks element-wise. Wrap each condition in parentheses because of operator precedence.

</details>

<details>
<summary><b>4. What is the index for?</b></summary>

It labels rows: fast lookups with `loc`, automatic alignment in arithmetic and joins, and meaningful row labels in results such as groupby output.

</details>

<details>
<summary><b>5. What is SettingWithCopyWarning?</b></summary>

A warning that you may be assigning into a temporary copy (chained indexing), so the original DataFrame might not change. Use a single `.loc[row_mask, col] = value`.

</details>

<details>
<summary><b>6. Why is apply slow?</b></summary>

It calls a Python function once per row or element, so it runs at Python-loop speed. Vectorised column operations run in compiled code.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Series and DataFrame
- [ ] I can explain: Loading and looking at data
- [ ] I can explain: A first look at any new dataset
- [ ] I can explain: Selecting columns and rows
- [ ] I can explain: Filtering and sorting
- [ ] I can explain: loc vs iloc: labels vs positions
- [ ] I can explain: Filtering, step by step
- [ ] I can explain: New columns, apply and the index
- [ ] I can explain: Changing data safely
- [ ] I can explain: Vectorise before you apply

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q17](#q17) loc vs iloc
- [ ] [Q18](#q18) Two Conditions
- [ ] [Q19](#q19) Best Subject per Student
- [ ] [Q20](#q20) Sort by Two Columns

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.